# Base modelling

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, confusion_matrix,
    accuracy_score, f1_score, recall_score, precision_score,
    average_precision_score, brier_score_loss
)

# ../ navigates up from notebooks/ to the repo root
df = pd.read_parquet("../data/engineered/parquet_saves/me_merged_base.parquet")
print(df.columns.tolist())

In [ ]:
id_cols = ['bank_id', 'country_iso', 'year']
# RT2 — build pre-crisis horizon target variable
# y_rt2 = 1 for T-1 and T-2 before each country's first crisis onset
# y_rt2 = NaN for all actual crisis years (excluded from train and test)
# y_rt2 = 0 for all tranquil years
# Relabeling is done once here on the full df; onset dates come from L&V (not future data).
df = df.copy()
df['crisis_rt2'] = 0.0

for _country in df['country_iso'].unique():
    _mask = df['country_iso'] == _country
    _crisis_yrs = df.loc[_mask & (df['crisis'] == 1), 'year']
    if len(_crisis_yrs) == 0:
        continue
    _onset = _crisis_yrs.min()
    # Label 2-year pre-crisis window as positive signal
    for _lag in [1, 2]:
        _pre = _onset - _lag
        df.loc[_mask & (df['year'] == _pre), 'crisis_rt2'] = 1.0
    # Mark all crisis years as NaN — excluded inside the expanding window loop
    df.loc[_mask & (df['crisis'] == 1), 'crisis_rt2'] = float('nan')

target = 'crisis_rt2'
print('RT2 target distribution (excl. NaN crisis years):')
print(df['crisis_rt2'].value_counts(dropna=True))
print(f"NaN rows (actual crisis years, excluded): {df['crisis_rt2'].isna().sum()}")
macro_cols = ['NGDP_RPCH','NGDPD', 'PCPIPCH', 'LUR', 'BCA_NGDPD', 'BIS_REER', 'RXF11FX_REVS', 'reer_g', 'fx_g']
micro_cols = [c for c in df.columns
              if c not in id_cols + [target, 'crisis'] + macro_cols]  # 'crisis' is not a feature
all_cols = micro_cols + macro_cols

print("Micro:", micro_cols)
print("Macro:", macro_cols)
print(f"{len(micro_cols)} micro, {len(macro_cols)} macro, {len(all_cols)} integrated features")

# Collinearity identification

## VIF

In [ ]:
def compute_vif(df, cols):
    tmp = df[cols].dropna() #selects only columns you want to check, and drops na since it cannot handle missing values
    vif_data = pd.DataFrame({
        'feature': cols,
        'VIF': [variance_inflation_factor(tmp.values, i) for i in range(len(cols))] #takes full feature matrix and index of column to check, then returns VIF score
    })
    return vif_data.sort_values('VIF', ascending=False)

In [ ]:
print("=== Micro ===")
print(compute_vif(df, micro_cols))

print("\n=== Macro ===")
print(compute_vif(df, macro_cols))

print("\n=== Integrated ===")
print(compute_vif(df, all_cols))

## Belsley collinearity diagnostics

In [ ]:
def colldiag(X): 
    """
    Belsley (1980) collinearity diagnostics — port of R's VisCollin::colldiag().

    Scales each column of X to unit length (standard Belsley preprocessing),
    then computes condition indices and variance decomposition proportions.

    Parameters
    ----------
    X : pd.DataFrame — feature matrix, no target column, no NaNs

    Returns
    -------
    pd.DataFrame with columns [CI, intercept, feat1, feat2, ...]
    Rows are components ordered from CI=1 (no collinearity) to largest CI.
    Flag: CI > 30 AND two variables both have VDP > 0.5 in same row → collinear pair.
    """
    
    #Intercept column
    arr = np.column_stack([np.ones(len(X)), X.values.astype(float)])
    col_names = ['intercept'] + list(X.columns)

    #Scale columns to unit length
    col_norms = np.sqrt((arr**2).sum(axis=0)) #L2 norm of each column
    col_norms[col_norms == 0] = 1 #avoid division by zero for constant columns
    arr = arr / col_norms

    #SVD, singular values returned in descending order
    _, s, Vt = np.linalg.svd(arr, full_matrices=False)
    V = Vt.T

    #Condition indices: largest singular value divided by each singular value'
    ci = s[0] / s # large n means near-zero singular value = near-collinearity

    #Variance decomposition proportions: proportion of variance of each variable in each component'
    numerators = (V ** 2) / (s ** 2) # variance of each variable in each component, scaled by singular values
    phi = numerators / numerators.sum(axis=1, keepdims=True) # normalize to get proportions

    #Output
    vdp = pd.DataFrame(phi.T, columns=col_names)
    vdp.insert(0, 'CI', ci)

    return vdp.round(3)

In [ ]:
def find_collinear_pairs(X, ci_threshold=30, vdp_threshold=0.5):
    """
    Runs Belsley diagnostics and returns collinear groups automatically.
    For each flagged row (CI > ci_threshold), finds non-intercept variables
    with VDP > vdp_threshold. Two or more in the same row = collinear group.
    The intercept is excluded by convention — Belsley always includes it
    but it should not be interpreted as part of a variable pair.
    """
    diag = colldiag(X)
    flagged = diag[diag['CI'] > ci_threshold]

    pairs = []
    for _, row in flagged.iterrows():
        suspects = [
            col for col in diag.columns
            if col not in ['CI', 'intercept'] and row[col] > vdp_threshold
        ]
        if len(suspects) >= 2:
            pairs.append({'CI': round(row['CI'], 1), 'variables': suspects})

    return pairs

In [ ]:
result = colldiag(df[micro_cols].dropna())
print(result[result['CI'] > 30])

In [ ]:
result = colldiag(df[macro_cols].dropna())
print(result[result['CI'] > 30])

In [ ]:
print("=== Micro — collinear pairs ===")
micro_pairs = find_collinear_pairs(df[micro_cols].dropna())
for pair in micro_pairs:
    print(f"  CI={pair['CI']}: {pair['variables']}")
if not micro_pairs:
    print("  None found")

print("\n=== Macro — collinear pairs ===")
macro_pairs = find_collinear_pairs(df[macro_cols].dropna())
for pair in macro_pairs:
    print(f"  CI={pair['CI']}: {pair['variables']}")
if not macro_pairs:
    print("  None found")

## Cols to drop based on automation

In [ ]:
# Drop decisions per feature set — determined by Belsley CI + VDP + economic reasoning.
# When running on a new dataset, call find_collinear_pairs() and record decisions here.
DROPS = {
    'micro': [
        # Accounting identities: exact linear dependence confirmed by CI > 1e10. can be dropped given this and derived ratios are unit free and more comparable than in their raw form
        'x_assets', 'x_equity', 'x_liab', 'x_liab_equity',
        # CI=211, VDP≈1.0 for both: keep x_loans_gross (less distorted pre-crisis by provisioning)
        'x_loans_net',
        # CI=34, VDP=0.946 vs 0.791: keep inc_op (absolute earnings level signal)
        'exp_total',
    ],
    'macro': [],  # fill after inspecting macro_pairs output above
}

In [ ]:
micro_cols_clean = [c for c in micro_cols if c not in DROPS['micro']]
macro_cols_clean = [c for c in macro_cols if c not in DROPS['macro']]
all_cols_clean   = [c for c in all_cols   if c not in DROPS['micro'] + DROPS['macro']]

print(f"Micro:      {len(micro_cols)} → {len(micro_cols_clean)} variables after drops")
print(f"Macro:      {len(macro_cols)} → {len(macro_cols_clean)} variables after drops")
print(f"Integrated: {len(all_cols)} → {len(all_cols_clean)} variables after drops")
print(f"\nMicro clean: {micro_cols_clean}")
print(f"Macro clean: {macro_cols_clean}")

Sanity check

In [ ]:
print("=== NaN counts in data===")
nan_counts = df[micro_cols_clean + macro_cols_clean].isnull().sum()
print(nan_counts[nan_counts > 0])
print("\nTotal rows:", len(df))

# Country ISO helper

In [ ]:
def get_country_col(df):
    # Different datasets use different column names — check both
    for col in ['country_iso', 'country_iso3']:
        if col in df.columns:
            return col
    # raise is OUTSIDE the loop — only fires if neither column is found
    raise ValueError(f"No country column found — expected 'country_iso' or 'country_iso3', got: {df.columns.tolist()}")

# Country dummies + feature set definitions

Country dummies are only fed into LR

In [ ]:
country_col = get_country_col(df)

# drop_first=True drops one country to avoid the dummy variable trap (last is implicit reference)
# astype(int) converts bool to 0/1 — required for SimpleImputer compatibility
country_dummies = pd.get_dummies(df[country_col], prefix='country', drop_first=True).astype(int)
country_dummy_cols = country_dummies.columns.tolist()

# Merge dummies into df so they are available when the expanding window splits df into train/test
# NOTE: re-run the splits cell after this one if you have already computed splits
df = pd.concat([df.reset_index(drop=True), country_dummies.reset_index(drop=True)], axis=1)

# Feature sets — same columns for RF and XGBoost (tree models capture country heterogeneity implicitly)
feature_sets = {
    'micro':      micro_cols_clean,
    'macro':      macro_cols_clean,
    'integrated': all_cols_clean,
}

# LR gets country dummies added to give country-specific intercepts (random intercept per country)
feature_sets_lr = {
    name: cols + country_dummy_cols
    for name, cols in feature_sets.items()
}

print(f"Country column: '{country_col}' — {len(country_dummy_cols)} dummies (reference country dropped)")
print("\nFeature set sizes:")
for name in feature_sets:
    print(f"  {name}: {len(feature_sets[name])} features | LR: {len(feature_sets_lr[name])}")

# H3 — Interaction terms (LR only, integrated condition)

H3 tests whether the predictive contribution of micro-level indicators depends on prevailing
macroeconomic conditions. For LR, this is operationalised by adding explicit microÃ—macro
interaction terms to the integrated feature set (tree models capture interactions implicitly).

Three interactions are included, selected on economic theory:

1. **Capital adequacy Ã— GDP growth** (`c_eqratio Ã— NGDP_RPCH`):
   Capital buffers matter more in recessions. Thin equity when GDP contracts = disproportionate
   vulnerability (Barrell et al., 2010; Demyanyk & Hasan, 2010; GonzÃ¡lez-Hermosillo, 1999).

2. **Leverage Ã— Unemployment** (`c_leverage Ã— LUR`):
   Rising unemployment drives loan defaults, hitting leveraged banks hardest. Macro timing
   (unemployment) Ã— bank-specific amplifier (leverage) = the trigger mechanism described in
   Citterio (2024) and GonzÃ¡lez-Hermosillo (1999).

3. **Loan concentration Ã— Inflation** (`r_loans_assets Ã— PCPIPCH`):
   Inflation erodes borrower repayment capacity (Du Plessis, 2022). Banks with high loan-to-asset
   ratios absorb the full impact of inflation-driven credit quality deterioration (Betz et al., 2012).

In [ ]:
# H3 interaction analysis not applicable for RT2 robustness check
# Placeholders so the Excel save cell doesn't fail
h3_df     = pd.DataFrame()
h3_auroc  = pd.Series(dtype=float, name='auroc')
int_coefs = pd.DataFrame()

# Expanding window split

In [ ]:
def expanding_window_split(df, min_train_years):
    years = sorted(df['year'].unique())
    splits = []

    for i, test_year in enumerate(years):
        if i < min_train_years:
            continue
        train = df[df['year'] < test_year]
        test = df[df['year'] == test_year]
        splits.append((train, test))
    return splits

In [ ]:
splits = expanding_window_split(df, min_train_years=5)
for train, test in splits[:3]:
    print(f"Train: {train['year'].min()}-{train['year'].max()}, Test: {test['year'].values[0]}")

# Results dataframe

In [ ]:
results = pd.DataFrame(columns = [
    'model', 'variant', 'dataset', 'test_year',
    'auroc', 'auprc', 'brier', 'best_threshold',
    'tp', 'fp', 'fn', 'tn',
    'f1', 'recall', 'precision',
    'miss_rate', 'false_alarm_rate', 'accuracy',
    'loss_07', 'loss_08', 'loss_09'
])

# Run_model helper

In [ ]:
def run_model(model, X_train, y_train, X_test, y_test, metadata, sample_weight=None, return_proba=False):
    # sample_weight is only used by MLP -- LR/RF/XGBoost handle imbalance via class_weight/scale_pos_weight
    fit_kwargs = {'sample_weight': sample_weight} if sample_weight is not None else {}
    model.fit(X_train, y_train, **fit_kwargs)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Find best threshold by maximising F1
    # np.where evaluates both branches eagerly, so the divide runs even where denom=0.
    # errstate suppresses the warning; np.where then correctly replaces those positions with 0.0.
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
    denom     = precisions + recalls
    with np.errstate(invalid='ignore', divide='ignore'):
        f1_scores = np.where(denom > 0, 2 * precisions * recalls / denom, 0.0)
    test_optimal_threshold = thresholds[np.argmax(f1_scores[:-1])]

    # A1: the alarm cut-off has to come from a year that was not used for evaluation. RT2 has a
    # single test year (2007) with no evaluated predecessor, so no honest cut-off exists. The
    # threshold-dependent metrics are left empty, exactly as 11a-d do for the first RT1 window.
    # test_optimal_threshold is kept for transparency: it is the best case, not a usable rule.
    best_threshold      = np.nan
    threshold_from_year = None
    tn = fp = fn = tp   = np.nan

    auroc            = roc_auc_score(y_test, y_proba)
    auprc            = average_precision_score(y_test, y_proba)
    brier            = brier_score_loss(y_test, y_proba)
    f1 = recall = precision = miss_rate = false_alarm_rate = accuracy = np.nan
    loss_07 = loss_08 = loss_09 = np.nan

    result = {
        **metadata,
        'auroc':            auroc,
        'auprc':            auprc,
        'brier':            brier,
        'best_threshold':        best_threshold,
        'threshold_from_year':   threshold_from_year,
        'test_optimal_threshold': test_optimal_threshold,
        'tp':               tp,
        'fp':               fp,
        'fn':               fn,
        'tn':               tn,
        'f1':               f1,
        'recall':           recall,
        'precision':        precision,
        'miss_rate':        miss_rate,
        'false_alarm_rate': false_alarm_rate,
        'accuracy':         accuracy,
        'loss_07':          loss_07,
        'loss_08':          loss_08,
        'loss_09':          loss_09,
    }

    # Only include raw probabilities when requested -- used by the main training loop
    # to populate proba_store for DeLong tests. Diagnostic loops (H3, no-dummies)
    # don't request them so their DataFrames stay clean.
    if return_proba:
        result['_y_proba'] = list(y_proba)
        result['_y_true']  = list(y_test)

    return result

# Training loop

In [ ]:
import optuna
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
import lightgbm as lgb

# Suppress per-trial Optuna output — only warnings and errors will print
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
def get_objective(model_name, X_tr_raw, y_tr, X_vl_raw, y_vl, n_neg_ratio):
    """
    Returns an Optuna objective function for the given model type.

    Imputation (and scaling for LR/MLP) are done inside each trial, fitted on inner_train only.
    This mirrors the full pipeline and prevents any leakage from inner_val into inner_train.
    Objective: maximise AUROC on the inner validation set.

    Parameters
    ----------
    X_tr_raw, X_vl_raw : pd.DataFrame — raw (pre-imputed) feature matrices
    y_tr, y_vl         : pd.Series   — crisis labels
    n_neg_ratio        : float        — n_neg / n_pos in the full training window (for XGBoost/MLP)
    """
    def objective(trial):
        # Fit imputer on inner_train only — same leakage rule as the outer loop
        imputer = SimpleImputer(strategy='median')
        X_tr = imputer.fit_transform(X_tr_raw)
        X_vl = imputer.transform(X_vl_raw)

        if model_name == 'logistic_regression':
            C = trial.suggest_float('C', 1e-4, 10.0, log=True)
            # Scale inside trial — fit on inner_train only
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_vl = scaler.transform(X_vl)
            # L2 (Ridge): shrinks all coefficients toward zero but keeps all features.
            # Chosen over L1 because we want a consistent feature set across model classes
            # for cross-model SHAP comparison — L1 would zero out different features each window.
            # lbfgs is the recommended solver for L2; faster and more stable than saga for small-medium data.
            model = LogisticRegression(
                solver='lbfgs', C=C,
                max_iter=1000, class_weight='balanced', random_state=7
            )

        elif model_name == 'random_forest':
            model = RandomForestClassifier(
                n_estimators=trial.suggest_int('n_estimators', 50, 500),
                max_depth=trial.suggest_int('max_depth', 3, 20),
                min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 20),
                max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
                class_weight='balanced', random_state=7,
            )

        elif model_name == 'xgboost':
            model = XGBClassifier(
                n_estimators=trial.suggest_int('n_estimators', 50, 500),
                max_depth=trial.suggest_int('max_depth', 2, 10),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                subsample=trial.suggest_float('subsample', 0.5, 1.0),
                min_child_weight=trial.suggest_int('min_child_weight', 1, 10),
                scale_pos_weight=n_neg_ratio,
                eval_metric='logloss', random_state=7
            )

        elif model_name == 'mlp':
            # MLP needs scaled inputs — gradient descent is sensitive to feature magnitude
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_vl = scaler.transform(X_vl)
            # suggest_categorical only accepts primitives — encode arch as string, decode to tuple
            arch = trial.suggest_categorical('hidden_layer_sizes', ['32', '64', '128', '64_32', '128_64'])
            # single-layer options (32/64/128): small to large capacity for limited panel data
            # two-layer funnels (64_32/128_64): standard progressive narrowing for tabular finance
            hidden_layer_sizes = {'32': (32,), '64': (64,), '128': (128,), '64_32': (64, 32), '128_64': (128, 64)}[arch]
            model = MLPClassifier(
                hidden_layer_sizes=hidden_layer_sizes,
                learning_rate_init=trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
                alpha=trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
                max_iter=5000,
                tol=1e-3,  # relaxed from default 1e-4 — suppresses warnings on flat loss landscapes without affecting AUROC
                random_state=7
            )

        elif model_name == 'decision_tree':
            model = DecisionTreeClassifier(
            max_depth=trial.suggest_int('max_depth', 2, 15),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 30),
            class_weight='balanced', random_state=7
            )
    
        elif model_name == 'lightgbm':
            # Ensure consistent feature names for fit and predict
            # SimpleImputer may return a DataFrame in newer sklearn versions
            if not isinstance(X_tr, pd.DataFrame):
                X_tr = pd.DataFrame(X_tr, columns=X_tr_raw.columns)
                X_vl = pd.DataFrame(X_vl, columns=X_vl_raw.columns)
            model = lgb.LGBMClassifier(
            n_estimators=trial.suggest_int('n_estimators', 50, 500),
            max_depth=trial.suggest_int('max_depth', 2, 10),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            num_leaves=trial.suggest_int('num_leaves', 15, 63),
            subsample=trial.suggest_float('subsample', 0.5, 1.0),
            min_child_samples=trial.suggest_int('min_child_samples', 5, 30),
            scale_pos_weight=n_neg_ratio,
            random_state=7, verbose=-1
        )

        # MLP has no class_weight parameter — pass sample_weight to handle class imbalance
        # All other models already handle imbalance via class_weight / scale_pos_weight
        if model_name == 'mlp':
            sw = np.where(y_tr == 1, n_neg_ratio, 1.0)
            model.fit(X_tr, y_tr, sample_weight=sw)
        else:
            model.fit(X_tr, y_tr)
        y_proba = model.predict_proba(X_vl)[:, 1]
        return roc_auc_score(y_vl, y_proba)
        
    return objective

## What's not tuned (fixed by design)


Each model has parameters that are **held fixed** — they are design choices that apply across all windows and variants, not tuning targets.

| Model | Fixed parameter | Value | Why fixed |
|-------|----------------|-------|-----------|
| All | `random_state` | `42` | Reproducibility across runs |
| LR | `solver` | `lbfgs` | Best for L2 penalty; faster and more stable than `saga` on small-medium panel data |
| LR | `penalty` | `l2` | Chosen for SHAP cross-model comparability — L1 would zero out different features per window, making SHAP values incomparable |
| LR | `max_iter` | `1000` | Sufficient for convergence; avoids `ConvergenceWarning` |
| LR, RF, DT | `class_weight` | `'balanced'` | Imbalance correction — a modelling principle, not a tuning target. Sklearn computes inverse-frequency weights automatically |
| XGBoost, LightGBM | `scale_pos_weight` | `n_neg / n_pos` | Same rationale as `class_weight`; XGB/LGBM use this scalar instead. Computed per window so it tracks the actual class ratio in each training set |
| XGBoost | `eval_metric` | `'logloss'` | Standard for binary classification with `predict_proba` |
| MLP | `activation` | `'relu'` (default) | Near-universal default for tabular data; no theoretical reason to tune it here |
| MLP | `max_iter` | `5000` | Generous budget to allow convergence on small crisis datasets without inflating wall time |
| MLP | `tol` | `1e-3` | Relaxed from sklearn default (`1e-4`) — suppresses warnings on flat loss landscapes in short inner windows; does not affect AUROC |
| MLP | `sample_weight` | `n_neg / n_pos` | MLP has no `class_weight` param — sample weights replicate the same imbalance correction used by the other models |
| DT | `criterion` | `'gini'` (default) | Gini and entropy give near-identical splits in practice; not worth a tuning dimension |
| LightGBM | `verbose` | `-1` | Suppress per-tree console output only — no effect on training |

In [ ]:
MODEL_NAMES = ['logistic_regression', 'random_forest', 'xgboost', 'mlp', 'decision_tree', 'lightgbm']
N_TRIALS = 50  # Optuna trials per model per window

# Two variants:
#   baseline_t1— full training window, no exclusions
#   rt1        — exclude crisis year (t) and post-crisis year (t+1) from training, per country.
#                Non-crisis countries keep all their data even in crisis calendar years.
#                Rationale: crisis/post-crisis bank profiles are "disorderly adjustments"
#                not informative about pre-crisis vulnerability (Bussiere & Fratzscher 2006).
#                With duration labeling, RT1 targets the ONSET year + t+1 only (not the full
#                duration), because the sustained crisis period is itself informative.
VARIANTS = [('rt2_2yr', False)]  # Single variant: pre-crisis horizon (2-year window)

all_results     = []
best_params_log = {}  # keyed by (variant, dataset, test_year, model)
proba_store     = {}  # (model, dataset, test_year, variant) → (y_true_list, y_proba_list)
fi_store        = {}  # (model, dataset, test_year, variant) → {feature: importance} (None for MLP)

for variant_name, _ in VARIANTS:
    for dataset_name, feat_cols in feature_sets.items():
        lr_cols = feature_sets_lr[dataset_name]
        print(f"\n=== Variant: {variant_name} | Dataset: {dataset_name} ===")

        # One-shot fixed split — train on 2004-2006, test on 2007
        # crisis_rt2 = 1 in 2006 (T-2) and 2007 (T-1) for onset-2008 countries
        # NaN crisis years dropped; non-crisis country banks serve as negatives
        # Single-element list keeps model training code below unchanged
        _train = df[df['year'] <= 2006].dropna(subset=[target])
        _test  = df[df['year'] == 2007].dropna(subset=[target])

        for train_full, test in [(_train, _test)]:
            test_year = 2007
            y_test    = test[target]

            if y_test.sum() == 0:
                print(f"ERROR: no pre-crisis signal in test year {test_year} — check crisis_rt2")
                continue

            train = train_full

            y_train     = train[target]
            n_pos       = int((y_train == 1).sum())
            n_neg       = int((y_train == 0).sum())
            n_neg_ratio = n_neg / n_pos if n_pos > 0 else 1.0

            print(f"  Train: years {sorted(train['year'].unique())} | n={len(train)} | crisis_rt2=1: {n_pos}")
            print(f"  Test : year=2007 | n={len(test)} | crisis_rt2=1: {int(y_test.sum())}")

            # No Optuna tuning — 3-year training window too thin for meaningful inner split
            # Default hyperparameters used; this is a proof-of-concept pre-crisis check
            for model_name in MODEL_NAMES:
                cols = lr_cols if model_name == 'logistic_regression' else feat_cols
                best_params = {}
                best_params_log[(variant_name, dataset_name, test_year, model_name)] = best_params

                # --- Outer loop: refit on full (filtered) training window ---
                imputer     = SimpleImputer(strategy='median')
                X_train_imp = imputer.fit_transform(train[cols])
                X_test_imp  = imputer.transform(test[cols])

                if model_name == 'logistic_regression':
                    scaler        = StandardScaler()
                    X_train_final = scaler.fit_transform(X_train_imp)
                    X_test_final  = scaler.transform(X_test_imp)
                    model = LogisticRegression(
                        solver='lbfgs',
                        C=best_params.get('C', 1.0),
                        max_iter=1000, class_weight='balanced', random_state=7
                    )

                elif model_name == 'random_forest':
                    X_train_final = X_train_imp
                    X_test_final  = X_test_imp
                    model = RandomForestClassifier(
                        n_estimators=best_params.get('n_estimators', 100),
                        max_depth=best_params.get('max_depth', None),
                        min_samples_leaf=best_params.get('min_samples_leaf', 1),
                        max_features=best_params.get('max_features', 'sqrt'),
                        class_weight='balanced', random_state=7
                    )

                elif model_name == 'xgboost':
                    X_train_final = X_train_imp
                    X_test_final  = X_test_imp
                    model = XGBClassifier(
                        n_estimators=best_params.get('n_estimators', 100),
                        max_depth=best_params.get('max_depth', 6),
                        learning_rate=best_params.get('learning_rate', 0.1),
                        subsample=best_params.get('subsample', 1.0),
                        min_child_weight=best_params.get('min_child_weight', 1),
                        scale_pos_weight=n_neg_ratio,
                        eval_metric='logloss', random_state=7
                    )

                elif model_name == 'mlp':
                    scaler        = StandardScaler()
                    X_train_final = scaler.fit_transform(X_train_imp)
                    X_test_final  = scaler.transform(X_test_imp)
                    arch = best_params.get('hidden_layer_sizes', '128')
                    hidden_layer_sizes = {'32': (32,), '64': (64,), '128': (128,), '64_32': (64, 32), '128_64': (128, 64)}.get(arch, (64,))
                    model = MLPClassifier(
                        hidden_layer_sizes=hidden_layer_sizes,
                        learning_rate_init=best_params.get('learning_rate_init', 1e-3),
                        alpha=best_params.get('alpha', 1e-4),
                        max_iter=5000,
                        tol=1e-3,  # relaxed from default 1e-4 — suppresses warnings on flat loss landscapes without affecting AUROC
                        random_state=7
                    )

                elif model_name == 'decision_tree':
                    X_train_final = X_train_imp
                    X_test_final  = X_test_imp
                    model = DecisionTreeClassifier(
                        max_depth=best_params.get('max_depth', 5),
                        min_samples_leaf=best_params.get('min_samples_leaf', 5),
                        class_weight='balanced', random_state=7
                    )

                elif model_name == 'lightgbm':
                    # Wrap as DataFrame: ensures consistent feature names for fit and predict
                    X_train_final = pd.DataFrame(X_train_imp, columns=cols)
                    X_test_final  = pd.DataFrame(X_test_imp,  columns=cols)
                    model = lgb.LGBMClassifier(
                        n_estimators=best_params.get('n_estimators', 100),
                        max_depth=best_params.get('max_depth', 6),
                        learning_rate=best_params.get('learning_rate', 0.1),
                        num_leaves=best_params.get('num_leaves', 31),
                        subsample=best_params.get('subsample', 1.0),
                        min_child_samples=best_params.get('min_child_samples', 20),
                        scale_pos_weight=n_neg_ratio,
                        random_state=7, verbose=-1
                    )

                sw = np.where(y_train == 1, n_neg_ratio, 1.0) if model_name == 'mlp' else None
                result = run_model(
                    model, X_train_final, y_train, X_test_final, y_test,
                    metadata={
                        'model':     model_name,
                        'variant':   variant_name,
                        'dataset':   dataset_name,
                        'test_year': test_year,
                    },
                    sample_weight=sw,
                    return_proba=True,
                )
                # Store raw probabilities for DeLong test, then remove from result
                # so they don't appear as columns in the results DataFrame
                proba_store[(model_name, dataset_name, test_year, variant_name)] = (
                    result.pop('_y_true'), result.pop('_y_proba')
                )
                # Tree models expose feature_importances_, LR exposes coef_
                # MLP has no native importance (None here; use SHAP in NB13 if needed)
                if hasattr(model, 'feature_importances_'):
                    fi_store[(model_name, dataset_name, test_year, variant_name)] = dict(zip(cols, model.feature_importances_))
                elif hasattr(model, 'coef_'):
                    fi_store[(model_name, dataset_name, test_year, variant_name)] = dict(zip(cols, model.coef_[0]))
                else:
                    fi_store[(model_name, dataset_name, test_year, variant_name)] = None
                all_results.append(result)

if not all_results:
    print("\nWARNING: no results collected — every test year had zero crisis observations.")
    print(f"Crisis counts by year:\n{df.groupby('year')[target].sum()}")
else:
    results = pd.DataFrame(all_results)
    print(f"\nCompleted: {len(results)} model-window evaluations")
    print(f"\nMean AUROC by model and dataset (rt2_2yr):")
    print(
        results[results['variant'] == 'rt2_2yr']
        .groupby(['model', 'dataset'])['auroc'].mean().round(3).unstack()
    )

# LR sanity check

# LR without country dummies — diagnostic

In [ ]:
# === LR without country dummies ===
# Not applicable for RT2 one-shot check:
#   - comparing dummy vs no-dummy requires multiple test years to be meaningful
#   - the single 2007 test fold is too narrow for this diagnostic
# Placeholders so the Excel save and DeLong cells don't fail
nodummy_results = []
nodummy_df      = pd.DataFrame()
comparison      = pd.DataFrame()

In [ ]:
# === LR Sanity Check ===
# (1) AUROC by test_year for LR (RT2 has only one: 2007)
# (2) Refit LR on the fixed training split and inspect coefficients

# (1) AUROC by year
lr_results = results[(results['model'] == 'logistic_regression') & (results['variant'] == 'rt2_2yr')]
print("=== LR AUROC by test_year and dataset (rt2_2yr) ===")
print(lr_results.pivot_table(index='test_year', columns='dataset', values='auroc').round(3).to_string())

# (2) Refit LR on fixed train split (2004-2006) — same split used in main loop
# dropna removes NaN crisis years from last_train so sklearn doesn't complain
last_train = df[df['year'] <= 2006].dropna(subset=[target])
last_test  = df[df['year'] == 2007].dropna(subset=[target])
last_year  = 2007
cols_inspect = feature_sets_lr['integrated']

imputer_inspect        = SimpleImputer(strategy='median')
X_train_imp_inspect    = imputer_inspect.fit_transform(last_train[cols_inspect])
scaler_inspect         = StandardScaler()
X_train_scaled_inspect = scaler_inspect.fit_transform(X_train_imp_inspect)

C_inspect = best_params_log.get(('rt2_2yr', 'integrated', last_year, 'logistic_regression'), {}).get('C', 1.0)

lr_inspect = LogisticRegression(
    solver='lbfgs', C=C_inspect,
    max_iter=1000, class_weight='balanced', random_state=7
)
lr_inspect.fit(X_train_scaled_inspect, last_train[target])

coef_df = pd.DataFrame({
    'feature': cols_inspect,
    'coef':    lr_inspect.coef_[0]
}).sort_values('coef', key=abs, ascending=False)

n_nonzero = (coef_df['coef'] != 0).sum()
print(f"\n=== LR coefficients — fixed split (test=2007), integrated, C={C_inspect:.4f} ===")
print(f"Non-zero: {n_nonzero} / {len(coef_df)} features")
print(coef_df.head(20).to_string(index=False))

## DeLong Test

In [ ]:
from scipy import stats as scipy_stats

def delong_test(y_true, y_score_a, y_score_b):
    """
    DeLong et al. (1988) non-parametric test for comparing two AUROCs.
    Returns (z, p): positive z means AUROC_a > AUROC_b; p is two-sided.
    Returns (nan, nan) when fewer than 2 positives or 2 negatives exist
    (covariance cannot be estimated with a single observation).

    Method: placement values (structural components) measure how well each
    positive score outranks each negative score. Their covariance across the
    two score vectors gives the variance of the AUROC difference — no parametric
    assumptions about score distributions needed.
    """
    y_true    = np.asarray(y_true,    dtype=int)
    y_score_a = np.asarray(y_score_a, dtype=float)
    y_score_b = np.asarray(y_score_b, dtype=float)

    pos = y_true == 1
    neg = y_true == 0
    m   = pos.sum()  # number of positives
    n   = neg.sum()  # number of negatives

    if m < 2 or n < 2:
        return np.nan, np.nan

    def placements(scores):
        s_pos = scores[pos]
        s_neg = scores[neg]
        # V10[i]: fraction of negatives outranked by positive i (ties split 50/50)
        V10 = np.array([(p > s_neg).mean() + 0.5 * (p == s_neg).mean() for p in s_pos])
        # V01[j]: fraction of positives that outrank negative j (ties split 50/50)
        V01 = np.array([(s_pos > nj).mean() + 0.5 * (s_pos == nj).mean() for nj in s_neg])
        return V10, V01

    V10_a, V01_a = placements(y_score_a)
    V10_b, V01_b = placements(y_score_b)

    auc_a = V10_a.mean()
    auc_b = V10_b.mean()

    # 2Ã—2 covariance matrices of the placement value pairs
    S10 = np.cov(np.vstack([V10_a, V10_b]), ddof=1)
    S01 = np.cov(np.vstack([V01_a, V01_b]), ddof=1)
    S   = S10 / m + S01 / n  # covariance matrix of (AUC_a, AUC_b)

    # Var(AUC_a - AUC_b) = S[0,0] + S[1,1] - 2*S[0,1]
    var_diff = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    if var_diff <= 0:
        return np.nan, np.nan

    z = (auc_a - auc_b) / np.sqrt(var_diff)
    p = 2 * scipy_stats.norm.sf(abs(z))
    return float(z), float(p)

In [ ]:
# === DeLong analysis ===
# Primary use: test H1 — does the integrated model significantly outperform micro-only
# and macro-only? Positive z = integrated AUROC > comparison AUROC.
# Secondary use: RT1 robustness — does excluding onset years change performance significantly?
# p < 0.05 in the RT1 vs baseline comparison would mean the onset rows were inflating results.

delong_rows = []
test_years  = sorted(results['test_year'].unique())

for variant_name in ['rt2_2yr']:
    for model_name in MODEL_NAMES:
        for test_year in test_years:

            # H1: integrated vs micro and integrated vs macro
            for label, cmp_dataset in [('integrated_vs_micro', 'micro'),
                                        ('integrated_vs_macro', 'macro')]:
                key_int = (model_name, 'integrated',  test_year, variant_name)
                key_cmp = (model_name, cmp_dataset,   test_year, variant_name)
                if key_int not in proba_store or key_cmp not in proba_store:
                    continue
                y_true, y_proba_int = proba_store[key_int]
                _,      y_proba_cmp = proba_store[key_cmp]
                z, p = delong_test(y_true, y_proba_int, y_proba_cmp)
                delong_rows.append({
                    'comparison': label,
                    'variant':    variant_name,
                    'model':      model_name,
                    'test_year':  test_year,
                    'z':          z,
                    'p':          p,
                    'sig_05':     bool(p < 0.05) if not np.isnan(p) else False,
                })


delong_df = pd.DataFrame(delong_rows)

# Summary: mean z and % significant windows per (comparison, variant, model)
print("=== DeLong summary: mean z-statistic and % significant test windows ===")
summary = (
    delong_df
    .groupby(['comparison', 'variant', 'model'])
    .agg(mean_z=('z', 'mean'), pct_sig=('sig_05', 'mean'), n=('z', 'count'))
    .round(3)
)
print(summary.to_string())

# Year-by-year p-values for the main H1 comparison (integrated vs micro, baseline)
print("\n=== H1: p-values by test year — integrated vs micro (baseline_t1) ===")
h1_mic = delong_df[
    (delong_df['comparison'] == 'integrated_vs_micro') &
    (delong_df['variant']    == 'rt2_2yr')
].pivot_table(index='test_year', columns='model', values='p').round(3)
print(h1_mic.to_string())
print("(p < 0.05 = integrated significantly outperforms micro-only in that year)")

# RT1 robustness summary: significant change means onset rows were inflating baseline results
print("\n=== RT1 robustness: % of windows where baseline vs rt1 differ significantly ===")
rt1_check = delong_df[delong_df['variant'] == 'rt1_check']
print(rt1_check.groupby(['model', 'comparison'])['sig_05'].mean().round(3).to_string())
print("(high % = onset-year exclusion materially changes predictions → baseline results were inflated)")

## Save all results

In [ ]:
# ---------------------------------------------------------------------------
# A46 falsification baseline
#
# The audit's objection to RT2 is that the labels are constant within a country and that the
# training labels already encode the 2008 onset. If that is right, a rule that ignores every
# feature and simply repeats the label the country carried in training should score close to a
# perfect AUROC on the test year. Reported here so the model AUROCs above can be read against it.
# ---------------------------------------------------------------------------
_test_year = 2007
_train = df[(df['year'] < _test_year) & (df[target].notna())]
_test  = df[(df['year'] == _test_year) & (df[target].notna())]

# the label each country carried in the training window
_country_label = _train.groupby('country_iso')[target].max()
_score = _test['country_iso'].map(_country_label).fillna(0.0).to_numpy(dtype=float)
_y     = _test[target].to_numpy(dtype=float)

_auroc = roc_auc_score(_y, _score)
_auprc = average_precision_score(_y, _score)

falsification = pd.DataFrame([{
    'rule': "copy the country's training label",
    'test_year': _test_year,
    'n_test': len(_test),
    'positives': int(_y.sum()),
    'auroc': _auroc,
    'auprc': _auprc,
}])
print(falsification.to_string(index=False))
print()
print(f"Best model AUROC in this notebook: {results['auroc'].max():.3f}")
print(f"Country-label rule AUROC:          {_auroc:.3f}")
print()
print("Reading: the pre-crisis relabelling can be solved from country identity alone, so the "
      "model AUROCs above measure how well the models recover which country is in the sample, "
      "not whether a crisis can be anticipated. RT2 is reported as a negative result.")


In [ ]:
import os
import pandas as pd

out_dir = "../outputs/results/rt2a"
os.makedirs(out_dir, exist_ok=True)

with pd.ExcelWriter(f"{out_dir}/results_rt2a_final.xlsx", engine="openpyxl") as writer:

    # --- Main results ---
    results.to_excel(writer, sheet_name="all_results", index=False)
    falsification.to_excel(writer, sheet_name="a46_falsification", index=False)

    results[results["variant"] == "rt2_2yr"].pivot_table(
        index="model", columns="dataset", values="auroc", aggfunc="mean"
    ).round(3).to_excel(writer, sheet_name="auroc_pivot_baseline")

    results[results["variant"] == "rt2_2yr"].groupby(["model", "dataset"])[
        ["auroc", "f1", "recall", "precision", "miss_rate", "false_alarm_rate",
  "loss_08"]
    ].mean().round(3).to_excel(writer, sheet_name="mean_metrics_baseline")


    # --- LR diagnostics ---
    comparison.to_excel(writer, sheet_name="lr_dummy_comparison")

    nodummy_df.to_excel(writer, sheet_name="lr_no_dummies_raw", index=False)

    coef_df.to_excel(writer, sheet_name="lr_coefficients", index=False)

    # --- H3 analysis ---

    # --- DeLong ---
    delong_df.to_excel(writer, sheet_name="delong_raw", index=False)

    summary.to_excel(writer, sheet_name="delong_summary")

    h1_mic.to_excel(writer, sheet_name="delong_h1_pvalues")

print(f"Saved to {out_dir}/results_rt2a_final.xlsx")

# Save proba_store — raw (y_true, y_proba) per model/window; needed for PR/ROC curves in NB13
with open(f"{out_dir}/proba_store_final.pkl", "wb") as f:
    pickle.dump(proba_store, f)
print(f"[SAVED] {out_dir}/proba_store_final.pkl")

# Save best_params_log — hyperparameter choices per model/window; for methodology table in NB13
with open(f"{out_dir}/best_params_log_final.pkl", "wb") as f:
    pickle.dump(best_params_log, f)
print(f"[SAVED] {out_dir}/best_params_log_final.pkl")

# Convert fi_store to long-format DataFrame and save; needed for feature importance plots in NB13
fi_rows = []
for (mdl, ds, yr, var), fi in fi_store.items():
    if fi is None:
        continue
    for feat, imp in fi.items():
        fi_rows.append({"model": mdl, "dataset": ds, "test_year": yr,
                        "variant": var, "feature": feat, "importance": imp})
fi_df = pd.DataFrame(fi_rows)
fi_df.to_parquet(f"{out_dir}/fi_store_final.parquet")
print(f"[SAVED] {out_dir}/fi_store_final.parquet")